# Stage 3 — SQL, Data Warehouse & OLAP Operations
## Cyber Crime Analytics for National Security

**Course:** Data Analytics & Visualization / Data Mining  
**Focus:** Dimensional Modeling, Star Schema, SQLite Warehouse, Analytical SQL, OLAP Operations (Roll-Up, Drill-Down, Slice, Dice), and Analytical Views  

---

### 1. Purpose & Analytical Scope
This notebook demonstrates the relational and dimensional modeling stage of the project lifecycle. Using validated Indian cybercrime data (NCRB 2023 Master Table and Rajya Sabha Historical Trends), we:
1. Initialize a star-schema analytical database in SQLite (`data/database/cybercrime.db`).
2. Define clear fact-table grains and conform dimensions (`dim_state`, `dim_year`, `dim_crime_category`, `dim_motive`).
3. Enforce mathematical double-counting safeguards using category leaf metadata (`is_leaf`).
4. Execute analytical SQL queries with aggregations, multi-table joins, grouped filtering (`HAVING`), and rankings.
5. Formally demonstrate OLAP operations: **Roll-Up**, **Drill-Down**, **Slice**, and **Dice** (using explicit demonstration parameters).
6. Query material views for downstream EDA and Power BI dashboards.
7. Verify referential integrity and complete reconciliation against official figures.

In [2]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_sqlite_connection, query_db
from src.db_builder import build_database, run_data_quality_checks

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
print('Environment and dependencies initialized successfully.')

Environment and dependencies initialized successfully.


### 2. Database Build & Schema Verification
We build/connect to `data/database/cybercrime.db` and inspect the dimension and fact tables.

In [4]:
# Build and initialize the database
conn = get_sqlite_connection()

# Query SQLite Master Schema to list all tables and views
schema_objects = query_db("""
    SELECT type, name 
    FROM sqlite_master 
    WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name;
""")
schema_objects

### 3. Dimension Tables Inspection
Let us inspect the dimensions populated in the database.

In [6]:
# Inspect States and Union Territories dimension
df_state = query_db('SELECT * FROM dim_state LIMIT 10;')
print('dim_state (Sample):')
display(df_state)

# Inspect Crime Category Dimension with Hierarchical Metadata
df_cat = query_db('SELECT category_id, category_display_name, act_group, is_leaf, section_reference FROM dim_crime_category LIMIT 12;')
print('dim_crime_category (Sample):')
display(df_cat)

dim_state (Sample):
   state_id                                state_name  is_ut
0         1               Andaman and Nicobar Islands      1
1         2                            Andhra Pradesh      0
2         3                         Arunachal Pradesh      0
3         4                                     Assam      0
4         5                                     Bihar      0
5         6                                Chandigarh      1
6         7                              Chhattisgarh      0
7         8  Dadra and Nagar Haveli and Daman and Diu      1
8         9                                     Delhi      1
9        10                                       Goa      0
dim_crime_category (Sample):
    category_id                                                                                                                                                                category_display_name act_group  is_leaf section_reference
0             1                               

### 4. Basic Aggregations, Grouping, and Rankings
Demonstrating standard SQL analytical querying: state rankings, national totals, and grouped filtering.

In [8]:
# Top 10 States by Cyber Crime Volume in 2023 (Filtering is_leaf = 1 to prevent double-counting)
top_states = query_db("""
    SELECT 
        s.state_name,
        s.is_ut,
        SUM(f.cases) AS total_cases,
        ROUND(100.0 * SUM(f.cases) / (
            SELECT SUM(f2.cases) 
            FROM fact_cybercrime_category_2023 f2 
            JOIN dim_crime_category c2 ON f2.category_id = c2.category_id 
            WHERE c2.is_leaf = 1
        ), 2) AS national_share_pct
    FROM fact_cybercrime_category_2023 f
    JOIN dim_state s ON f.state_id = s.state_id
    JOIN dim_crime_category c ON f.category_id = c.category_id
    WHERE c.is_leaf = 1
    GROUP BY s.state_id, s.state_name, s.is_ut
    ORDER BY total_cases DESC
    LIMIT 10;
""")
top_states

In [9]:
# Grouped Filtering with HAVING: States with > 2,000 cases and count of active crime categories
high_burden_states = query_db("""
    SELECT 
        s.state_name,
        SUM(f.cases) AS total_cases,
        COUNT(DISTINCT CASE WHEN f.cases > 0 THEN f.category_id END) AS active_crime_categories
    FROM fact_cybercrime_category_2023 f
    JOIN dim_state s ON f.state_id = s.state_id
    JOIN dim_crime_category c ON f.category_id = c.category_id
    WHERE c.is_leaf = 1
    GROUP BY s.state_id, s.state_name
    HAVING SUM(f.cases) > 2000
    ORDER BY total_cases DESC;
""")
high_burden_states

### 5. OLAP Operations Demonstration

> **Note on Demonstration Parameters:** Specific dimension values selected below (e.g. `State = 'Telangana'`, `Motive = 'Fraud'`, dynamic Top 4 states) are **demonstration selections** designed to illustrate OLAP querying mechanics.

#### 5.1 Roll-Up (Aggregation Hierarchy)
Aggregating finer level metrics up the dimension hierarchy: **Leaf Category → Act Group → All-India Total**.

In [11]:
# OLAP Roll-Up: Summarizing cases by Legal Act Group
rollup_act = query_db("""
    SELECT 
        c.act_group,
        COUNT(DISTINCT c.category_id) AS leaf_category_count,
        SUM(f.cases) AS total_cases,
        ROUND(100.0 * SUM(f.cases) / (
            SELECT SUM(f_all.cases) 
            FROM fact_cybercrime_category_2023 f_all 
            JOIN dim_crime_category c_all ON f_all.category_id = c_all.category_id 
            WHERE c_all.is_leaf = 1
        ), 2) AS share_pct
    FROM fact_cybercrime_category_2023 f
    JOIN dim_crime_category c ON f.category_id = c.category_id
    WHERE c.is_leaf = 1
    GROUP BY c.act_group
    ORDER BY total_cases DESC;
""")
rollup_act

#### 5.2 Drill-Down (Navigating to Detailed Granularity)
**Demonstration Selection:** Navigating from state totals down to specific crime categories and legal sections for State = `Telangana`.

In [13]:
# OLAP Drill-Down: Detailed breakdown of IT Act crimes in Telangana
drilldown_state = query_db("""
    SELECT 
        s.state_name,
        c.act_group,
        c.category_display_name,
        c.section_reference,
        f.cases,
        ROUND(100.0 * f.cases / NULLIF(state_act.act_total, 0), 2) AS pct_of_state_it_act
    FROM fact_cybercrime_category_2023 f
    JOIN dim_state s ON f.state_id = s.state_id
    JOIN dim_crime_category c ON f.category_id = c.category_id
    JOIN (
        SELECT f3.state_id, SUM(f3.cases) AS act_total
        FROM fact_cybercrime_category_2023 f3
        JOIN dim_crime_category c3 ON f3.category_id = c3.category_id
        WHERE c3.is_leaf = 1 AND c3.act_group = 'IT Act'
        GROUP BY f3.state_id
    ) state_act ON s.state_id = state_act.state_id
    WHERE s.state_name = 'Telangana' 
      AND c.act_group = 'IT Act' 
      AND c.is_leaf = 1
    ORDER BY f.cases DESC
    LIMIT 8;
""")
drilldown_state

#### 5.3 Slice (Fixing One Dimension)
**Demonstration Selection:** Isolating a single slice of the data cube where `Motive = 'Fraud'` across top reporting states.

In [15]:
# OLAP Slice: Slicing on Motive = Fraud across top reporting states
slice_motive = query_db("""
    SELECT 
        s.state_name,
        m.motive_display_name,
        f.motive_count AS fraud_cases,
        ROUND(100.0 * f.motive_count / (
            SELECT SUM(f_fr.motive_count) 
            FROM fact_cybercrime_motive_2023 f_fr 
            JOIN dim_motive m_fr ON f_fr.motive_id = m_fr.motive_id 
            WHERE m_fr.motive_raw_name = 'motive__Fraud'
        ), 2) AS national_fraud_share_pct
    FROM fact_cybercrime_motive_2023 f
    JOIN dim_state s ON f.state_id = s.state_id
    JOIN dim_motive m ON f.motive_id = m.motive_id
    WHERE m.motive_raw_name = 'motive__Fraud'
    ORDER BY f.motive_count DESC
    LIMIT 8;
""")
slice_motive

#### 5.4 Dice (Multi-Dimensional Sub-Cube)
**Multi-Dimensional Sub-Cube** constrained across:  
- **Geography:** Dynamically calculated Top 4 States by 2023 total volume (via subquery)  
- **Crime Types:** Financial / Fraud / Cheating Categories  
- **Time:** Year 2023

In [17]:
# OLAP Dice: Multi-dimensional sub-cube with dynamic Top 4 state subquery
dice_query = query_db("""
    WITH top_4_states AS (
        SELECT f_sub.state_id
        FROM fact_cybercrime_category_2023 f_sub
        JOIN dim_crime_category c_sub ON f_sub.category_id = c_sub.category_id
        WHERE c_sub.is_leaf = 1
        GROUP BY f_sub.state_id
        ORDER BY SUM(f_sub.cases) DESC
        LIMIT 4
    )
    SELECT 
        s.state_name,
        c.category_display_name,
        f.cases
    FROM fact_cybercrime_category_2023 f
    JOIN dim_state s ON f.state_id = s.state_id
    JOIN dim_crime_category c ON f.category_id = c.category_id
    JOIN top_4_states t4 ON s.state_id = t4.state_id
    WHERE (
        c.category_display_name LIKE '%Fraud%'
        OR c.category_display_name LIKE '%Cheating%'
        OR c.category_display_name LIKE '%Identity Theft%'
    )
    AND c.is_leaf = 1
    ORDER BY s.state_name, f.cases DESC;
""")
dice_query.head(15)

### 6. Analytical Views & Power BI Preparation
We query the registered analytical views designed for executive dashboards and downstream analysis.

In [19]:
# View 1: State Cyber Crime Summary (Top 5)
vw_state = query_db('SELECT * FROM vw_state_cybercrime_summary ORDER BY reported_grand_total DESC LIMIT 5;')
display(vw_state)

# View 6: Historical Trend & YoY Growth (Sample for Maharashtra & Karnataka)
vw_trend = query_db("""
    SELECT state_name, year, cases, prev_year_cases, yoy_case_change, yoy_growth_pct
    FROM vw_historical_trend_growth
    WHERE state_name IN ('Karnataka', 'Maharashtra')
    ORDER BY state_name, year;
""")
display(vw_trend)

   state_id     state_name  is_ut  reported_grand_total  total_leaf_cases  reconciliation_diff  it_act_cases  it_act_share_pct  ipc_cases  ipc_share_pct  sll_cases  sll_share_pct  total_motives_reported
0        16      Karnataka      0                 21889             21889                    0         21870             99.91         18           0.08          1           0.00                   21889
1        32      Telangana      0                 18236             18236                    0           439              2.41      17787          97.54         10           0.05                   18236
2        34  Uttar Pradesh      0                 10794             10794                    0         10102             93.59        689           6.38          3           0.03                   10794
3        21    Maharashtra      0                  8103              8103                    0           960             11.85       7131          88.00         12           0.15          

### 7. Data Quality & Referential Integrity Verification
Let us run the automated integrity checks to verify zero foreign key violations and 100% mathematical reconciliation.

In [21]:
# Execute comprehensive data quality check suite
run_data_quality_checks(conn)
conn.close()


--- DATA QUALITY & INTEGRITY CHECKS ---
[PASS] Foreign Key Integrity: 0 violations.
  Table `dim_state`: 36 rows
  Table `dim_year`: 6 rows
  Table `dim_crime_category`: 49 rows
  Table `dim_motive`: 19 rows
  Table `fact_cybercrime_category_2023`: 1,764 rows
  Table `fact_cybercrime_motive_2023`: 684 rows
  Table `fact_cybercrime_trend`: 180 rows
[PASS] All dimension and fact row counts match expected exact dimensions.
[PASS] Leaf Category Sum Reconciliation: 100% match across all 36 States/UTs (0 discrepancies).
  View `vw_state_cybercrime_summary`: 36 rows returned successfully.
  View `vw_category_cybercrime_summary`: 49 rows returned successfully.
  View `vw_act_group_summary`: 3 rows returned successfully.
  View `vw_state_category_analysis`: 1,764 rows returned successfully.
  View `vw_motive_summary`: 19 rows returned successfully.
  View `vw_historical_trend_growth`: 180 rows returned successfully.
[PASS] All analytical views operational.


### 8. Academic Interpretations & Limitations

#### Key Takeaways:
1. **Dimensional Design:** The star-schema cleanly separates dimension attributes (`dim_state`, `dim_year`, `dim_crime_category`, `dim_motive`) from quantitative measures (`fact_cybercrime_category_2023`, `fact_cybercrime_motive_2023`, `fact_cybercrime_trend`).
2. **Double-Counting Prevention:** By flagging leaf categories (`is_leaf = 1`), additive queries remain mathematically sound and accurately sum to official state grand totals.
3. **National Concentration:** Fraud is by far the leading motive (>68% of all motives), and IT Act offences account for over 51% of national cases, heavily concentrated in states like Karnataka, Telangana, and Maharashtra.

#### Methodological Limitations:
- **Temporal Granularity:** Detailed category-wise breakdowns are currently available cross-sectionally for 2023; the 2018–2022 dataset provides aggregate state counts and is maintained as a separate trend fact table.